# Lunar Sex Addendum (Legacy)

**Source script:** `eda_lunar_sex_addendum.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import logging
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from scipy.stats import chi2 as scipy_chi2_dist
except Exception:
    scipy_chi2_dist = None


ROOT = Path(__file__).resolve().parent
ENRICHED_PATH = ROOT / "outputs" / "imputed_dataset_enriched.csv"
ENRICHED_XLSX_PATH = ROOT / "outputs" / "imputed_dataset_enriched.xlsx"
OUT_DIR = ROOT / "outputs" / "eda_v2" / "lunar_sex"

MIN_BUCKET_N = 20
PHASE_ORDER = ["new", "waxing", "full", "waning"]


@dataclass
class MoonPhaseInfo:
    column: str
    is_numeric: bool
    mapping_note: str


def setup_logging() -> logging.Logger:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )
    return logging.getLogger("eda_lunar_sex_addendum")


def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ""
    s = str(value).strip().lower()
    s = (
        s.replace("ä", "ae")
        .replace("ö", "oe")
        .replace("ü", "ue")
        .replace("ß", "ss")
    )
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def load_data() -> pd.DataFrame:
    if ENRICHED_PATH.exists():
        return pd.read_csv(ENRICHED_PATH)
    if ENRICHED_XLSX_PATH.exists():
        return pd.read_excel(ENRICHED_XLSX_PATH)
    raise FileNotFoundError(
        f"Could not find enriched dataset at {ENRICHED_PATH} or {ENRICHED_XLSX_PATH}"
    )


def detect_column(columns: List[str], keywords: List[str]) -> Optional[str]:
    best_col = None
    best_score = -1
    for col in columns:
        cn = normalize_text(col)
        score = 0
        for kw in keywords:
            if normalize_text(kw) in cn:
                score += 2
        if cn in {normalize_text(k) for k in keywords}:
            score += 3
        if score > best_score:
            best_score = score
            best_col = col
    if best_score <= 0:
        return None
    return best_col


def detect_date_column(df: pd.DataFrame) -> Optional[str]:
    preferred = [
        "presentation_date",
        "date of presentation",
        "admission_date",
        "date",
    ]
    cols = list(df.columns)
    by_name = {normalize_text(c): c for c in cols}
    for p in preferred:
        if p in cols:
            return p
        pn = normalize_text(p)
        if pn in by_name:
            return by_name[pn]
    candidates = [c for c in cols if "date" in normalize_text(c)]
    return candidates[0] if candidates else None


def detect_moon_phase_column(df: pd.DataFrame) -> MoonPhaseInfo:
    cols = list(df.columns)
    preferred = [
        "moon_phase_fraction_illuminated",
        "moon_phase_angle_deg",
        "moon_phase_category",
    ]
    by_norm = {normalize_text(c): c for c in cols}

    for p in preferred:
        p_norm = normalize_text(p)
        if p in cols:
            col = p
        elif p_norm in by_norm:
            col = by_norm[p_norm]
        else:
            continue
        is_num = pd.to_numeric(df[col], errors="coerce").notna().sum() > 0
        return MoonPhaseInfo(
            column=col,
            is_numeric=is_num,
            mapping_note=f"Detected moon column: {col} ({'numeric' if is_num else 'categorical'})",
        )

    moon_like = [c for c in cols if "moon" in normalize_text(c) and "phase" in normalize_text(c)]
    if not moon_like:
        moon_like = [c for c in cols if "moon" in normalize_text(c)]
    if not moon_like:
        raise ValueError("No moon phase column detected in dataset.")

    col = moon_like[0]
    is_num = pd.to_numeric(df[col], errors="coerce").notna().sum() > 0
    return MoonPhaseInfo(
        column=col,
        is_numeric=is_num,
        mapping_note=f"Detected moon column: {col} ({'numeric' if is_num else 'categorical'})",
    )


def moon_phase_from_date(dt: pd.Timestamp) -> str:

    ref_new_moon = pd.Timestamp("2000-01-06")
    synodic_days = 29.53058867
    delta_days = (dt.normalize() - ref_new_moon).days
    phase = (delta_days % synodic_days) / synodic_days

    if phase < 0.125 or phase >= 0.875:
        return "new"
    if phase < 0.375:
        return "waxing"
    if phase < 0.625:
        return "full"
    return "waning"


def moon_category_from_text(value: object) -> str:
    s = normalize_text(value)
    if s in {"", "nan", "none", "unknown", "unbekannt", "?"}:
        return "unknown"
    if "new" in s:
        return "new"
    if "full" in s:
        return "full"
    if "wan" in s:
        return "waning"
    if "wax" in s:
        return "waxing"
    if "first quarter" in s or "first" in s:
        return "waxing"
    if "last quarter" in s or "third quarter" in s or "last" in s:
        return "waning"
    if "crescent" in s:

        return "unknown"
    if "gibbous" in s:
        return "unknown"
    return sanitize_category(s)


def sanitize_category(value: object) -> str:
    s = normalize_text(value)
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s if s else "unknown"


def derive_moon_phase_series(df: pd.DataFrame, info: MoonPhaseInfo) -> pd.Series:
    raw = df[info.column]
    if info.is_numeric:
        num = pd.to_numeric(raw, errors="coerce")
        labels = pd.Series(index=df.index, dtype="object")
        valid = num.dropna()
        if valid.nunique() >= 2:
            try:
                binned = pd.qcut(valid, q=4, labels=PHASE_ORDER, duplicates="drop")
                labels.loc[valid.index] = binned.astype(str)
            except ValueError:
                labels.loc[valid.index] = np.nan
        return labels.fillna("unknown")
    return raw.map(sanitize_category).fillna("unknown")


def compute_exposure_days(date_series: pd.Series, case_phase_series: pd.Series) -> Tuple[pd.DataFrame, str]:
    dt = pd.to_datetime(date_series, errors="coerce").dropna()
    if dt.empty:
        raise ValueError("No valid presentation dates found for calendar exposure calculation.")
    phases_no_unknown = set(case_phase_series.dropna().astype(str).unique()) - {"unknown"}

    if phases_no_unknown.issubset(set(PHASE_ORDER)):
        dmin = dt.min().normalize()
        dmax = dt.max().normalize()
        cal = pd.date_range(dmin, dmax, freq="D")
        phase = pd.Series([moon_phase_from_date(d) for d in cal], index=cal)
        exposure = (
            phase.value_counts()
            .rename_axis("moon_phase")
            .reset_index(name="exposure_days")
        )
        exposure["moon_phase"] = pd.Categorical(exposure["moon_phase"], categories=PHASE_ORDER, ordered=True)
        exposure = exposure.sort_values("moon_phase").reset_index(drop=True)
        return exposure, "calendar_based"



    tmp = pd.DataFrame(
        {
            "date": pd.to_datetime(date_series, errors="coerce"),
            "moon_phase": case_phase_series.astype("object"),
        }
    ).dropna(subset=["date"])
    tmp["date"] = tmp["date"].dt.normalize()
    exposure = (
        tmp.groupby("moon_phase", dropna=False)["date"]
        .nunique()
        .rename("exposure_days")
        .reset_index()
    )
    exposure["moon_phase"] = exposure["moon_phase"].fillna("unknown").astype(str)
    exposure = exposure.sort_values("moon_phase").reset_index(drop=True)
    return exposure, "observed_date_fallback"


def poisson_rate_ci(count: float, exposure_days: float, alpha: float = 0.05) -> Tuple[float, float]:
    if exposure_days <= 0:
        return (np.nan, np.nan)
    rate = (count / exposure_days) * 100.0
    if count <= 0:
        upper = (1.96 / exposure_days) * 100.0
        return (0.0, max(0.0, upper))
    se = (math.sqrt(count) / exposure_days) * 100.0
    z = 1.96
    return (max(0.0, rate - z * se), max(0.0, rate + z * se))


def rr_ci_poisson(rr: float, count: float, total_count: float) -> Tuple[float, float]:
    if rr <= 0 or count <= 0 or total_count <= 0:
        return (np.nan, np.nan)
    se_log_rr = math.sqrt((1.0 / count) + (1.0 / total_count))
    z = 1.96
    low = math.exp(math.log(rr) - z * se_log_rr)
    high = math.exp(math.log(rr) + z * se_log_rr)
    return (low, high)


def build_lunar_tables(
    case_phase_series: pd.Series,
    exposure_days_df: pd.DataFrame,
) -> Dict[str, pd.DataFrame]:
    counts = (
        case_phase_series.fillna("unknown")
        .value_counts()
        .rename_axis("moon_phase")
        .reset_index(name="case_count")
    )
    phase_order = list(exposure_days_df["moon_phase"].astype(str).unique())
    for p in case_phase_series.fillna("unknown").astype(str).unique():
        if p not in phase_order:
            phase_order.append(p)
    all_phases = pd.DataFrame({"moon_phase": phase_order})
    counts = all_phases.merge(counts, on="moon_phase", how="left").fillna({"case_count": 0})

    merged = all_phases.merge(exposure_days_df, on="moon_phase", how="left").merge(counts, on="moon_phase", how="left")
    merged["exposure_days"] = merged["exposure_days"].fillna(0)
    merged["case_count"] = merged["case_count"].fillna(0)

    merged["rate_per_100_days"] = np.where(
        merged["exposure_days"] > 0,
        100.0 * merged["case_count"] / merged["exposure_days"],
        np.nan,
    )
    ci_vals = merged.apply(
        lambda r: poisson_rate_ci(float(r["case_count"]), float(r["exposure_days"])),
        axis=1,
    )
    merged["rate_ci_low_95"] = [v[0] for v in ci_vals]
    merged["rate_ci_high_95"] = [v[1] for v in ci_vals]

    total_cases = float(merged["case_count"].sum())
    total_exposure = float(merged["exposure_days"].sum())
    baseline_rate = (100.0 * total_cases / total_exposure) if total_exposure > 0 else np.nan
    merged["baseline_rate_per_100_days"] = baseline_rate
    merged["relative_risk_vs_baseline"] = merged["rate_per_100_days"] / baseline_rate

    rr_ci_vals = merged.apply(
        lambda r: rr_ci_poisson(
            rr=float(r["relative_risk_vs_baseline"]) if pd.notna(r["relative_risk_vs_baseline"]) else np.nan,
            count=float(r["case_count"]),
            total_count=total_cases,
        ),
        axis=1,
    )
    merged["rr_ci_low_95"] = [v[0] for v in rr_ci_vals]
    merged["rr_ci_high_95"] = [v[1] for v in rr_ci_vals]

    return {
        "counts": merged[["moon_phase", "case_count"]].copy(),
        "rates": merged[
            [
                "moon_phase",
                "exposure_days",
                "case_count",
                "rate_per_100_days",
                "rate_ci_low_95",
                "rate_ci_high_95",
            ]
        ].copy(),
        "rr": merged[
            [
                "moon_phase",
                "case_count",
                "exposure_days",
                "rate_per_100_days",
                "baseline_rate_per_100_days",
                "relative_risk_vs_baseline",
                "rr_ci_low_95",
                "rr_ci_high_95",
            ]
        ].copy(),
    }


def compute_chi_square(case_counts_df: pd.DataFrame, exposure_days_df: pd.DataFrame) -> Dict[str, float]:
    merged = (
        exposure_days_df.merge(case_counts_df, on="moon_phase", how="left")
        .fillna({"case_count": 0})
        .copy()
    )
    total_cases = float(merged["case_count"].sum())
    total_exposure = float(merged["exposure_days"].sum())
    merged["expected"] = np.where(
        total_exposure > 0,
        total_cases * (merged["exposure_days"] / total_exposure),
        np.nan,
    )
    valid = merged[(merged["expected"] > 0) & merged["expected"].notna()].copy()
    chi_square = float(((valid["case_count"] - valid["expected"]) ** 2 / valid["expected"]).sum()) if not valid.empty else np.nan
    dfree = max(0, len(valid) - 1)
    if scipy_chi2_dist is not None and pd.notna(chi_square):
        p_value = float(scipy_chi2_dist.sf(chi_square, dfree))
    else:
        p_value = np.nan
    return {
        "chi_square": chi_square,
        "degrees_of_freedom": float(dfree),
        "p_value": p_value,
        "total_cases": total_cases,
        "total_exposure_days": total_exposure,
    }


def detect_sex_and_repro_columns(df: pd.DataFrame) -> Tuple[Optional[str], Optional[str]]:
    cols = list(df.columns)
    sex_col = detect_column(cols, ["sex", "gender", "geschlecht"])
    repro_col = detect_column(
        cols,
        ["neuter", "neutered", "castrat", "kastr", "spayed", "steril", "reproductive"],
    )
    return sex_col, repro_col


def normalize_sex(value: object) -> str:
    t = normalize_text(value)
    if t in {"", "unknown", "unbekannt", "nan", "none", "?", "na", "n a"}:
        return "unknown"
    female_tokens = {"female", "f", "weiblich", "w"}
    male_tokens = {"male", "m", "maennlich", "maennl", "mannlich"}

    if t in female_tokens:
        return "female"
    if t in male_tokens:
        return "male"

    toks = set(t.split())
    if toks & female_tokens:
        return "female"
    if toks & male_tokens:
        return "male"
    return "unknown"


def normalize_repro(value: object) -> str:
    t = normalize_text(value)
    if t in {"", "unknown", "unbekannt", "nan", "none", "?", "na", "n a"}:
        return "unknown"
    neutered_tokens = {
        "neutered",
        "spayed",
        "castrated",
        "kastriert",
        "sterilisiert",
        "yes",
        "y",
        "fixed",
    }
    intact_tokens = {
        "intact",
        "entire",
        "unkastriert",
        "nicht kastriert",
        "no",
        "n",
    }

    if t in neutered_tokens:
        return "neutered"
    if t in intact_tokens:
        return "intact"

    toks = set(t.split())
    if toks & neutered_tokens:
        return "neutered"
    if toks & intact_tokens:
        return "intact"
    return "unknown"


def build_bucket_columns(df: pd.DataFrame, sex_col: Optional[str], repro_col: Optional[str]) -> pd.DataFrame:
    out = df.copy()
    if sex_col is None:
        out["sex_norm"] = "unknown"
    else:
        out["sex_norm"] = out[sex_col].map(normalize_sex)

    if repro_col is None:
        out["repro_norm"] = "unknown"
    else:
        out["repro_norm"] = out[repro_col].map(normalize_repro)

    out["sex_repro_bucket"] = out["sex_norm"] + "_" + out["repro_norm"]
    all_buckets = [
        "female_neutered",
        "female_intact",
        "female_unknown",
        "male_neutered",
        "male_intact",
        "male_unknown",
        "unknown_neutered",
        "unknown_intact",
        "unknown_unknown",
    ]
    out["sex_repro_bucket"] = pd.Categorical(out["sex_repro_bucket"], categories=all_buckets, ordered=False)
    return out


def bucket_counts(df: pd.DataFrame) -> pd.DataFrame:
    total_n = len(df)
    c = (
        df["sex_repro_bucket"]
        .value_counts(dropna=False)
        .rename_axis("bucket")
        .reset_index(name="n")
    )
    c["n"] = c["n"].astype(int)
    c["pct"] = np.where(total_n > 0, 100.0 * c["n"] / total_n, np.nan)
    c["underpowered_n_lt_20"] = c["n"] < MIN_BUCKET_N
    return c


def analyze_buckets(
    df: pd.DataFrame,
    exposure_days_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object]]:
    results_rates: List[pd.DataFrame] = []
    results_rr: List[pd.DataFrame] = []
    counts_df = bucket_counts(df)

    non_zero = counts_df[counts_df["n"] > 0].copy()
    n_underpowered = int((non_zero["n"] < MIN_BUCKET_N).sum())
    collapse_needed = bool((len(non_zero) > 0) and (n_underpowered / max(1, len(non_zero)) > 0.5))
    if int((non_zero["n"] >= MIN_BUCKET_N).sum()) < 3:
        collapse_needed = True

    total_exposure = float(exposure_days_df["exposure_days"].sum())

    phase_order = list(exposure_days_df["moon_phase"].astype(str).unique())
    for bucket, sub in df.groupby("sex_repro_bucket", observed=False):
        bucket_name = str(bucket)
        n_bucket = len(sub)
        phases = sub["moon_phase_bin"].fillna("unknown")
        phase_counts = (
            phases.value_counts()
            .rename_axis("moon_phase")
            .reset_index(name="case_count")
        )
        all_phase = pd.DataFrame({"moon_phase": phase_order})
        phase_counts = all_phase.merge(phase_counts, on="moon_phase", how="left").fillna({"case_count": 0})

        merged = all_phase.merge(exposure_days_df, on="moon_phase", how="left").merge(phase_counts, on="moon_phase", how="left")
        merged["exposure_days"] = merged["exposure_days"].fillna(0)
        merged["case_count"] = merged["case_count"].fillna(0)
        merged["bucket"] = bucket_name
        merged["n_bucket"] = n_bucket
        merged["underpowered_n_lt_20"] = n_bucket < MIN_BUCKET_N

        merged["rate_per_100_days"] = np.where(
            merged["exposure_days"] > 0,
            100.0 * merged["case_count"] / merged["exposure_days"],
            np.nan,
        )
        ci_vals = merged.apply(
            lambda r: poisson_rate_ci(float(r["case_count"]), float(r["exposure_days"])),
            axis=1,
        )
        merged["rate_ci_low_95"] = [v[0] for v in ci_vals]
        merged["rate_ci_high_95"] = [v[1] for v in ci_vals]

        bucket_baseline = (100.0 * n_bucket / total_exposure) if total_exposure > 0 else np.nan
        merged["bucket_baseline_rate_per_100_days"] = bucket_baseline
        merged["rr_vs_bucket_baseline"] = merged["rate_per_100_days"] / bucket_baseline

        overall_baseline = (100.0 * len(df) / total_exposure) if total_exposure > 0 else np.nan
        merged["overall_baseline_rate_per_100_days"] = overall_baseline
        merged["rr_vs_overall_baseline"] = merged["rate_per_100_days"] / overall_baseline

        results_rates.append(
            merged[
                [
                    "bucket",
                    "n_bucket",
                    "underpowered_n_lt_20",
                    "moon_phase",
                    "exposure_days",
                    "case_count",
                    "rate_per_100_days",
                    "rate_ci_low_95",
                    "rate_ci_high_95",
                ]
            ].copy()
        )
        results_rr.append(
            merged[
                [
                    "bucket",
                    "n_bucket",
                    "underpowered_n_lt_20",
                    "moon_phase",
                    "case_count",
                    "rr_vs_bucket_baseline",
                    "rr_vs_overall_baseline",
                ]
            ].copy()
        )

    rate_df = pd.concat(results_rates, ignore_index=True) if results_rates else pd.DataFrame()
    rr_df = pd.concat(results_rr, ignore_index=True) if results_rr else pd.DataFrame()

    analysis_meta = {
        "collapse_to_sex": collapse_needed,
        "n_nonzero_buckets": int(len(non_zero)),
        "n_underpowered_nonzero_buckets": n_underpowered,
    }
    return rate_df, rr_df, analysis_meta


def collapse_to_sex(
    df: pd.DataFrame,
    exposure_days_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rates: List[pd.DataFrame] = []
    rrs: List[pd.DataFrame] = []
    total_exposure = float(exposure_days_df["exposure_days"].sum())
    total_cases = len(df)

    phase_order = list(exposure_days_df["moon_phase"].astype(str).unique())
    for sex in ["female", "male", "unknown"]:
        sub = df[df["sex_norm"] == sex]
        n = len(sub)
        phase_counts = (
            sub["moon_phase_bin"].fillna("unknown")
            .value_counts()
            .rename_axis("moon_phase")
            .reset_index(name="case_count")
        )
        all_phase = pd.DataFrame({"moon_phase": phase_order})
        merged = all_phase.merge(exposure_days_df, on="moon_phase", how="left").merge(phase_counts, on="moon_phase", how="left")
        merged["exposure_days"] = merged["exposure_days"].fillna(0)
        merged["case_count"] = merged["case_count"].fillna(0)
        merged["bucket"] = sex
        merged["n_bucket"] = n
        merged["underpowered_n_lt_20"] = n < MIN_BUCKET_N
        merged["rate_per_100_days"] = np.where(
            merged["exposure_days"] > 0, 100.0 * merged["case_count"] / merged["exposure_days"], np.nan
        )
        ci_vals = merged.apply(lambda r: poisson_rate_ci(float(r["case_count"]), float(r["exposure_days"])), axis=1)
        merged["rate_ci_low_95"] = [v[0] for v in ci_vals]
        merged["rate_ci_high_95"] = [v[1] for v in ci_vals]

        bucket_baseline = (100.0 * n / total_exposure) if total_exposure > 0 else np.nan
        overall_baseline = (100.0 * total_cases / total_exposure) if total_exposure > 0 else np.nan
        merged["rr_vs_bucket_baseline"] = merged["rate_per_100_days"] / bucket_baseline
        merged["rr_vs_overall_baseline"] = merged["rate_per_100_days"] / overall_baseline

        rates.append(
            merged[
                [
                    "bucket",
                    "n_bucket",
                    "underpowered_n_lt_20",
                    "moon_phase",
                    "exposure_days",
                    "case_count",
                    "rate_per_100_days",
                    "rate_ci_low_95",
                    "rate_ci_high_95",
                ]
            ].copy()
        )
        rrs.append(
            merged[
                [
                    "bucket",
                    "n_bucket",
                    "underpowered_n_lt_20",
                    "moon_phase",
                    "case_count",
                    "rr_vs_bucket_baseline",
                    "rr_vs_overall_baseline",
                ]
            ].copy()
        )

    return pd.concat(rates, ignore_index=True), pd.concat(rrs, ignore_index=True)


def plot_lunar_rates(rates_df: pd.DataFrame, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    phase_order = rates_df["moon_phase"].astype(str).tolist()
    x = np.arange(len(phase_order))
    vals = []
    lows = []
    highs = []
    for p in phase_order:
        row = rates_df[rates_df["moon_phase"] == p]
        if row.empty:
            vals.append(np.nan)
            lows.append(np.nan)
            highs.append(np.nan)
        else:
            vals.append(float(row.iloc[0]["rate_per_100_days"]))
            lows.append(float(row.iloc[0]["rate_ci_low_95"]))
            highs.append(float(row.iloc[0]["rate_ci_high_95"]))
    yerr_low = [max(0.0, v - l) if pd.notna(v) and pd.notna(l) else 0 for v, l in zip(vals, lows)]
    yerr_high = [max(0.0, h - v) if pd.notna(v) and pd.notna(h) else 0 for v, h in zip(vals, highs)]
    ax.bar(x, vals, color="#3B6FB6", width=0.7)
    ax.errorbar(x, vals, yerr=[yerr_low, yerr_high], fmt="none", ecolor="black", capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(phase_order)
    ax.set_ylabel("Rate per 100 days")
    ax.set_xlabel("Moon phase")
    ax.set_title("Exposure-normalized lunar rates")
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def plot_lunar_rate_by_sex(df: pd.DataFrame, exposure_days_df: pd.DataFrame, out_path: Path) -> None:
    total_exposure = float(exposure_days_df["exposure_days"].sum())
    vals = []
    labels = ["female", "male", "unknown"]
    for sex in labels:
        n = len(df[df["sex_norm"] == sex])
        vals.append((100.0 * n / total_exposure) if total_exposure > 0 else np.nan)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(labels, vals, color=["#D65F5F", "#4C78A8", "#8C8C8C"])
    ax.set_title("Lunar rate per 100 days by sex")
    ax.set_xlabel("Sex")
    ax.set_ylabel("Rate per 100 days")
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def plot_lunar_rate_by_sex_repro(
    counts_df: pd.DataFrame,
    exposure_days_df: pd.DataFrame,
    out_path: Path,
    sufficient: bool,
) -> None:
    fig, ax = plt.subplots(figsize=(11, 5))
    total_exposure = float(exposure_days_df["exposure_days"].sum())
    if not sufficient:
        ax.text(
            0.5,
            0.5,
            "Insufficient cell sizes for sex x reproductive grouped rate plot",
            ha="center",
            va="center",
        )
        ax.set_axis_off()
        fig.tight_layout()
        fig.savefig(out_path, dpi=180)
        plt.close(fig)
        return

    ordered = [
        "female_neutered",
        "female_intact",
        "female_unknown",
        "male_neutered",
        "male_intact",
        "male_unknown",
        "unknown_neutered",
        "unknown_intact",
        "unknown_unknown",
    ]
    plot_df = counts_df.set_index("bucket").reindex(ordered).fillna({"n": 0}).reset_index()
    plot_df["rate_per_100_days"] = np.where(total_exposure > 0, 100.0 * plot_df["n"] / total_exposure, np.nan)

    x = np.arange(len(plot_df))
    ax.bar(x, plot_df["rate_per_100_days"], color="#7A5195")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["bucket"], rotation=45, ha="right")
    ax.set_title("Lunar rate per 100 days by sex x reproductive status")
    ax.set_xlabel("Sex x reproductive status bucket")
    ax.set_ylabel("Rate per 100 days")
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def write_chi_square_txt(stats: Dict[str, float], out_path: Path) -> None:
    lines = [
        "Chi-square goodness-of-fit test (observed lunar case counts vs exposure-weighted expected)",
        f"chi_square: {stats['chi_square']}",
        f"degrees_of_freedom: {int(stats['degrees_of_freedom'])}",
        f"p_value: {stats['p_value']}",
        f"total_cases: {stats['total_cases']}",
        f"total_exposure_days: {stats['total_exposure_days']}",
    ]
    if scipy_chi2_dist is None:
        lines.append("note: scipy not available, p-value could not be computed.")
    out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_addendum_md(
    out_path: Path,
    moon_info: MoonPhaseInfo,
    date_col: Optional[str],
    sex_col: Optional[str],
    repro_col: Optional[str],
    analysis_meta: Dict[str, object],
) -> None:
    collapse_note = (
        "Collapsed to sex-only summary because too many sex x reproductive cells were underpowered."
        if analysis_meta.get("collapse_to_sex", False)
        else "Sex x reproductive status strata retained for analysis."
    )
    lines = [
        "# Lunar + Sex Stratified EDA Addendum",
        "",
        "## Inputs and Detection",
        f"- Moon phase source: `{moon_info.column}`",
        f"- Moon phase handling: {'numeric -> 4 quantile bins (new/waxing/full/waning)' if moon_info.is_numeric else 'categorical used directly'}",
        f"- Date column: `{date_col}`",
        f"- Sex column detected: `{sex_col}`",
        f"- Reproductive status column detected: `{repro_col}`",
        "",
        "## Methods Summary",
        "- Exposure-normalized lunar rates computed per 100 calendar days.",
        "- Exposure days estimated from daily calendar between min and max presentation date with astronomical moon phase approximation.",
        f"- Exposure method used in this run: `{analysis_meta.get('exposure_method', 'unknown')}`.",
        "- Relative risk computed against baseline rate; 95% CI uses Poisson approximation.",
        "- Chi-square goodness-of-fit compares observed lunar case counts with exposure-weighted expected counts.",
        "",
        "## Stratification Notes",
        f"- Non-zero sex x reproductive buckets: {analysis_meta.get('n_nonzero_buckets')}",
        f"- Underpowered non-zero buckets (n < {MIN_BUCKET_N}): {analysis_meta.get('n_underpowered_nonzero_buckets')}",
        f"- Collapse decision: {collapse_note}",
        "",
        "## Interpretation",
        "- This is exploratory EDA and hypothesis-generating.",
        "- Exposure normalization was applied to account for differing calendar-time lunar exposure.",
        "- Small cell sizes can destabilize rate and relative risk estimates in some strata.",
        "- No causal inference; hypothesis-generating only.",
        "",
    ]
    out_path.write_text("\n".join(lines), encoding="utf-8")


def main() -> None:
    logger = setup_logging()
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    df = load_data()
    logger.info("Loaded enriched dataset: shape=%s", df.shape)

    date_col = detect_date_column(df)
    if date_col is None:
        raise ValueError("No date column found; needed for exposure-normalized calendar rates.")
    df["__date__"] = pd.to_datetime(df[date_col], errors="coerce")
    if df["__date__"].isna().all():
        raise ValueError(f"Date column '{date_col}' contains no parseable dates.")

    moon_info = detect_moon_phase_column(df)
    logger.info(moon_info.mapping_note)
    df["moon_phase_bin"] = derive_moon_phase_series(df, moon_info)

    exposure_days_df, exposure_method = compute_exposure_days(df["__date__"], df["moon_phase_bin"])
    lunar_tables = build_lunar_tables(df["moon_phase_bin"], exposure_days_df)
    chi_stats = compute_chi_square(lunar_tables["counts"], exposure_days_df)


    lunar_tables["counts"].to_csv(OUT_DIR / "lunar_counts.csv", index=False)
    lunar_tables["rates"].to_csv(OUT_DIR / "lunar_rates.csv", index=False)
    lunar_tables["rr"].to_csv(OUT_DIR / "lunar_rr.csv", index=False)
    write_chi_square_txt(chi_stats, OUT_DIR / "lunar_chi_square.txt")
    plot_lunar_rates(lunar_tables["rates"], OUT_DIR / "lunar_rate_plot.png")


    sex_col, repro_col = detect_sex_and_repro_columns(df)
    df_b = build_bucket_columns(df, sex_col, repro_col)
    counts = bucket_counts(df_b)
    rates_by_bucket, rr_by_bucket, analysis_meta = analyze_buckets(df_b, exposure_days_df)


    if analysis_meta["collapse_to_sex"]:
        collapsed_rates, collapsed_rr = collapse_to_sex(df_b, exposure_days_df)
        collapsed_rates["analysis_level"] = "sex_collapsed"
        collapsed_rr["analysis_level"] = "sex_collapsed"
        rates_by_bucket["analysis_level"] = "sex_repro"
        rr_by_bucket["analysis_level"] = "sex_repro"
        rates_by_bucket = pd.concat([rates_by_bucket, collapsed_rates], ignore_index=True)
        rr_by_bucket = pd.concat([rr_by_bucket, collapsed_rr], ignore_index=True)
    else:
        rates_by_bucket["analysis_level"] = "sex_repro"
        rr_by_bucket["analysis_level"] = "sex_repro"


    counts.to_csv(OUT_DIR / "sex_repro_bucket_counts.csv", index=False)
    rates_by_bucket.to_csv(OUT_DIR / "lunar_rates_by_bucket.csv", index=False)
    rr_by_bucket.to_csv(OUT_DIR / "lunar_relative_risk_by_bucket.csv", index=False)


    plot_lunar_rate_by_sex(df_b, exposure_days_df, OUT_DIR / "lunar_rate_by_sex.png")
    sufficient_grouped = bool(((counts["n"] >= MIN_BUCKET_N) & (counts["n"] > 0)).sum() >= 2)
    plot_lunar_rate_by_sex_repro(
        counts_df=counts,
        exposure_days_df=exposure_days_df,
        out_path=OUT_DIR / "lunar_rate_by_sex_repro.png",
        sufficient=sufficient_grouped,
    )


    write_addendum_md(
        out_path=OUT_DIR / "lunar_sex_addendum.md",
        moon_info=moon_info,
        date_col=date_col,
        sex_col=sex_col,
        repro_col=repro_col,
        analysis_meta={**analysis_meta, "exposure_method": exposure_method},
    )

    logger.info("Saved lunar + sex addendum outputs to: %s", OUT_DIR)


if __name__ == "__main__":
    main()
